# Exercise: Iranian Web Logs with Spark RDDs

Use Apache Spark 3.5.9 RDDs to load Iranian web logs, parse each record, validate the results, perform basic analysis, and store the parsed data in HDFS.

Do not use pandas or Spark DataFrames.

## Learning objectives

By completing this exercise, you will practice:

- moving local Windows files into HDFS
- reading several files with `SparkContext.textFile`
- parsing semi-structured text with a compiled regular expression
- using `map`, `filter`, `reduceByKey`, and RDD actions
- tracking malformed records instead of silently dropping them
- storing parsed JSON Lines output in HDFS
- connecting input partitions to Spark tasks

## Dataset

The Windows dataset directory is:

```text
C:\data\weblogs
```

Inside WSL, the same directory is:

```text
/mnt/c/data/weblogs
```

This exercise expects the 20 source files under `C:\data\weblogs\parts`. Use the part files rather than a large combined copy, because both versions may contain the same events. Loading both would duplicate the data.

## 1. Start HDFS and Spark

Run these commands in a WSL terminal before starting the notebook:

```bash
start-dfs.sh
/opt/spark/sbin/start-master.sh
/opt/spark/sbin/start-worker.sh "spark://$(hostname):7077"
jps
```

Confirm that the HDFS daemons and the Spark `Master` and `Worker` are running. This exercise does not use YARN.

## 2. Inspect the local files

Do not begin parsing until you understand the available files and have inspected records from more than one part.

In [ ]:
%%bash
LOCAL_PARTS='/mnt/c/data/weblogs/parts'

test -d "$LOCAL_PARTS" || { echo "Missing directory: $LOCAL_PARTS"; exit 1; }
find "$LOCAL_PARTS" -maxdepth 1 -type f -name 'weblog-*.log' -printf '%f %s bytes\n' | sort -V
echo "Part-file count: $(find "$LOCAL_PARTS" -maxdepth 1 -type f -name 'weblog-*.log' | wc -l)"
head -2 "$LOCAL_PARTS/weblog-1.log"
head -2 "$LOCAL_PARTS/weblog-20.log"

Write down what each part of a record means. Inspect unusual records as well as the first row. Determine how missing values, quoted text, timestamps, request paths, status codes, response sizes, referrers, and user agents are represented.

## 3. Upload the raw part files to HDFS

Each student writes to `/user/$USER/iranian-weblogs`. The input directory contains only raw logs.

In [ ]:
%%bash
LOCAL_PARTS='/mnt/c/data/weblogs/parts'
HDFS_INPUT="/user/$USER/iranian-weblogs/input"

hdfs dfs -mkdir -p "$HDFS_INPUT"
hdfs dfs -put -f "$LOCAL_PARTS"/weblog-*.log "$HDFS_INPUT/"
hdfs dfs -ls -h "$HDFS_INPUT"

Verify that HDFS contains 20 files. If the count is different, stop and correct the input before continuing.

In [ ]:
%%bash
HDFS_INPUT="/user/$USER/iranian-weblogs/input"

FILE_COUNT=$(hdfs dfs -ls "$HDFS_INPUT" | awk '$1 ~ /^-/ {count++} END {print count+0}')
echo "HDFS input-file count: $FILE_COUNT"
if [ "$FILE_COUNT" -ne 20 ]; then
    echo 'Expected exactly 20 part files.'
    exit 1
fi


## 4. Connect to the Spark standalone master

In [ ]:
import os
import socket

from pyspark.sql import SparkSession

master_url = os.environ.get(
    "SPARK_MASTER",
    f"spark://{socket.gethostname()}:7077",
)

spark = (
    SparkSession.builder
    .appName("D283-Iranian-Weblogs-Exercise")
    .master(master_url)
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version :", spark.version)
print("Spark master  :", sc.master)
print("Application ID:", sc.applicationId)
print("Spark UI      :", sc.uiWebUrl)

## 5. Read every part file as one RDD

Passing the HDFS input directory to `textFile` reads all files inside it. Each RDD record is one log line. The call is lazy.

In [ ]:
hdfs_user = os.environ["USER"]
hdfs_input = f"hdfs:///user/{hdfs_user}/iranian-weblogs/input"
hdfs_parsed_output = f"hdfs:///user/{hdfs_user}/iranian-weblogs/output/parsed-json"
hdfs_rejected_output = f"hdfs:///user/{hdfs_user}/iranian-weblogs/output/rejected"

raw_logs_rdd = sc.textFile(hdfs_input)

print("Input path      :", hdfs_input)
print("Input partitions:", raw_logs_rdd.getNumPartitions())

Explain in your own words:

1. Why can the RDD have more than one partition?
2. How are the 20 files related to the input partitions?
3. During a stage, how is the partition count related to the number of tasks?

In [ ]:
sample_lines = raw_logs_rdd.take(10)

for line in sample_lines:
    print(line)

## 6. Document the schema you discover

Add a Markdown table below with one row per field. Include:

- field name
- meaning
- example value
- target Python type
- representation for a missing value

Your final parsed record should include every field present in a complete source record.

### Discovered schema

Replace this text with your schema table.

## 7. Build and test a compiled regular expression

Create one complete regular expression with named groups. Do not parse the row using only `split(' ')`; quoted request and user-agent fields can contain spaces.

A typical combined Apache record has this general shape:

```text
client_ip identity user [timestamp zone] "method path protocol" status bytes "referrer" "user_agent"
```

Use the actual data, not only this example, to determine the final pattern.

In [ ]:
import re

# TODO: replace the placeholder with your complete named-group pattern.
LOG_PATTERN = re.compile(r"TODO")

print("Named groups:", LOG_PATTERN.groupindex)

Test the pattern against several real rows and at least one malformed row. Write assertions for known field values.

In [ ]:
# TODO: test valid rows from different files.
# TODO: confirm that a deliberately malformed row does not match.
# TODO: assert values for several named groups.


## 8. Write one line parser

The parser must return a tagged pair:

```text
('valid', parsed_dictionary)
('rejected', original_line)
```

Convert status codes and response sizes to integers. Choose a consistent value for missing fields. Keep the timestamp text unless you have a clear reason to convert it at this stage.

In [ ]:
def parse_log_line(line):
    # TODO: apply LOG_PATTERN.
    # TODO: return ('rejected', line) when it does not match.
    # TODO: build a dictionary from the named groups.
    # TODO: convert numeric fields to integers.
    # TODO: return ('valid', record).
    raise NotImplementedError("Complete parse_log_line")

Test the parser locally on the small sample before applying it to the complete RDD.

In [ ]:
# TODO: parse sample_lines and inspect the returned tagged pairs.
# TODO: add assertions for record keys and Python types.


## 9. Parse the complete RDD

`map` runs `parse_log_line` on every source record. Cache the tagged RDD because the valid and rejected branches both reuse it.

In [ ]:
# Run this after completing and testing parse_log_line.
tagged_records_rdd = raw_logs_rdd.map(parse_log_line).cache()

In [ ]:
valid_records_rdd = (
    tagged_records_rdd
    .filter(lambda item: item[0] == "valid")
    .map(lambda item: item[1])
)

rejected_lines_rdd = (
    tagged_records_rdd
    .filter(lambda item: item[0] == "rejected")
    .map(lambda item: item[1])
)

## 10. Validate record counts

Valid plus rejected must equal the raw input count. This check proves that the pipeline did not silently lose records.

In [ ]:
raw_count = raw_logs_rdd.count()
valid_count = valid_records_rdd.count()
rejected_count = rejected_lines_rdd.count()

print(f"Raw records     : {raw_count:,}")
print(f"Valid records   : {valid_count:,}")
print(f"Rejected records: {rejected_count:,}")

assert valid_count + rejected_count == raw_count

In [ ]:
rejected_lines_rdd.take(10)

Investigate rejected records. A nonzero rejected count may reveal malformed source data or a pattern that is too strict. Do not change the pattern merely to force every row to match.

## 11. Required RDD analysis

Using only RDD transformations and actions, answer these questions:

1. How many requests use each HTTP method?
2. How many responses have each status code?
3. Which ten IP addresses made the most requests?
4. Which ten request paths were requested most often?
5. How many responses are client errors from 400 through 499?
6. How many responses are server errors from 500 through 599?
7. What are the earliest and latest event timestamps?
8. How many response bytes were transferred in total?

Use `map` and `reduceByKey` for grouped counts. Use actions such as `count`, `sum`, `min`, `max`, `collect`, or `takeOrdered` where appropriate.

In [ ]:
# TODO: requests by HTTP method


In [ ]:
# TODO: responses by status code


In [ ]:
# TODO: top ten IP addresses


In [ ]:
# TODO: top ten request paths


In [ ]:
# TODO: client errors, server errors, timestamp range, and total bytes


## 12. Convert parsed dictionaries to JSON Lines

JSON Lines stores one JSON object per line. It is suitable for semi-structured RDD output and preserves field names without a DataFrame schema.

In [ ]:
import json

parsed_json_rdd = valid_records_rdd.map(
    lambda record: json.dumps(record, ensure_ascii=False, sort_keys=True)
)

parsed_json_rdd.take(2)

## 13. Delete only the previous exercise outputs

Spark will not overwrite an existing output directory. The raw HDFS input is not deleted.

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/iranian-weblogs/output"

hdfs dfs -rm -r -f "$HDFS_OUTPUT/parsed-json"
hdfs dfs -rm -r -f "$HDFS_OUTPUT/rejected"

## 14. Save parsed and rejected records to HDFS

Each `saveAsTextFile` call is an action and creates a directory containing `part-*` files.

In [ ]:
sc.setJobGroup("save-parsed-weblogs", "Save parsed Iranian weblogs as JSON Lines")
parsed_json_rdd.saveAsTextFile(hdfs_parsed_output)

sc.setJobGroup("save-rejected-weblogs", "Save rejected Iranian weblog records")
rejected_lines_rdd.saveAsTextFile(hdfs_rejected_output)

print("Parsed output  :", hdfs_parsed_output)
print("Rejected output:", hdfs_rejected_output)

## 15. Verify the HDFS output

In [ ]:
%%bash
HDFS_OUTPUT="/user/$USER/iranian-weblogs/output"

hdfs dfs -ls -h "$HDFS_OUTPUT/parsed-json"
hdfs dfs -cat "$HDFS_OUTPUT/parsed-json"/part-* | head -3
echo '--- Rejected output ---'
hdfs dfs -ls -h "$HDFS_OUTPUT/rejected"

## 16. Inspect jobs, stages, tasks, and lineage

Open the Spark application UI printed earlier. For at least one analysis action and one save action, record:

- job ID
- number of stages
- tasks in each stage
- input size
- shuffle read and shuffle write, if present

Explain which transformation caused each shuffle.

In [ ]:
print(parsed_json_rdd.toDebugString().decode("utf-8"))

## Submission checklist

- All 20 part files were uploaded to the student's HDFS input directory.
- The schema is documented.
- The regular expression is compiled once and uses named groups.
- Numeric fields use appropriate Python numeric types.
- Valid and rejected counts reconcile with the raw count.
- Rejected records are inspected and saved.
- All required analysis questions are answered with RDD operations.
- Parsed records are valid JSON objects stored in HDFS.
- Spark UI observations and RDD lineage are documented.
- No pandas or DataFrame operations are used.

## 17. Stop Spark

Run this cell when the exercise is complete.

In [ ]:
tagged_records_rdd.unpersist()
spark.stop()
print("Spark session stopped.")